## Baseline

These tables contain multiple records for a single client.
Before merging with `application_train` or `application_test`, it is necessary
to engineer features and aggregate the tables to a single row per `SK_ID_CURR`.

Standard numerical features are aggregated in the same way (median, max).
Categorical features are transformed into separate COUNT and SHARE features.
Special features are handled manually based on their meaning.

### 1. Import all

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from narwhals import Categorical
from narwhals.selectors import categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, ConfusionMatrixDisplay, confusion_matrix, roc_curve
from lightgbm.callback import early_stopping, log_evaluation

import lightgbm as lgb

In [2]:
path_to_data = "/home/usl/PycharmProjects/home-credit-default-risk/data/raw/home-credit-default-risk/"
# path_to_data = "/content/drive/MyDrive/Home_Credit_data/"

application_train_df = pd.read_csv(path_to_data+"application_train.csv")
application_test_df = pd.read_csv(path_to_data+"application_test.csv")
bureau_df = pd.read_csv(path_to_data+"bureau.csv")
bureau_balance_df = pd.read_csv(path_to_data+"bureau_balance.csv")
credit_card_balance_df = pd.read_csv(path_to_data+"credit_card_balance.csv")
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
installments_payments_df = pd.read_csv(path_to_data+"installments_payments.csv")
POS_CASH_balance_df = pd.read_csv(path_to_data+"POS_CASH_balance.csv")
previous_application_df = pd.read_csv(path_to_data+"previous_application.csv")

### 2. Feature engineering

#### Main function for numerical and categorical features

### 2.1 Numerical features: main function

In [3]:
agg_func = ["median", "max"]
def agg_numerical_features(df, id_col, numerical_cols, prefix):
    tables = []

    for feat in numerical_cols:
        temp = (df.groupby(id_col)[feat].agg(agg_func))
        temp.columns = [prefix + "_" + feat + "_" + func for func in agg_func]
        tables.append(temp)
    result = pd.concat(tables, axis=1)
    return result

For example:

In [4]:
#numerical_cols_bureau_df = ["AMT_CREDIT_MAX_OVERDUE", "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT","AMT_CREDIT_SUM_LIMIT", "AMT_CREDIT_SUM_OVERDUE", "AMT_ANNUITY"]

In [5]:
#bureau_num = make_numerical_features(bureau_df, "SK_ID_CURR", numerical_cols_bureau_df, "BUREAU")
#bureau_num

,BUREAU_AMT_CREDIT_MAX_OVERDUE_median,BUREAU_AMT_CREDIT_MAX_OVERDUE_max,BUREAU_AMT_CREDIT_SUM_median,BUREAU_AMT_CREDIT_SUM_max,BUREAU_AMT_CREDIT_SUM_DEBT_median,BUREAU_AMT_CREDIT_SUM_DEBT_max,BUREAU_AMT_CREDIT_SUM_LIMIT_median,BUREAU_AMT_CREDIT_SUM_LIMIT_max,BUREAU_AMT_CREDIT_SUM_OVERDUE_median,BUREAU_AMT_CREDIT_SUM_OVERDUE_max,BUREAU_AMT_ANNUITY_median,BUREAU_AMT_ANNUITY_max
SK_ID_CURR,,,,,,,,,,,,
100001,NaN,NaN,168345.00,378000.00,0.000,373239.00,0.0,0.000,0.0,0.0,0.0,10822.5
100002,40.500,5043.645,54130.50,450000.00,0.000,245781.00,0.0,31988.565,0.0,0.0,0.0,0.0
100003,0.000,0.000,92576.25,810000.00,0.000,0.00,0.0,810000.000,0.0,0.0,NaN,NaN
100004,0.000,0.000,94518.90,94537.80,0.000,0.00,0.0,0.000,0.0,0.0,NaN,NaN
100005,0.000,0.000,58500.00,568800.00,25321.500,543087.00,0.0,0.000,0.0,0.0,0.0,4261.5
...,...,...,...,...,...,...,...,...,...,...,...,...
456249,0.000,18945.000,248692.50,765000.00,0.000,163071.00,0.0,0.000,0.0,0.0,NaN,NaN
456250,0.000,0.000,483349.50,2153110.05,391731.615,1840308.48,0.0,58268.385,0.0,0.0,51799.5,384147.0
456253,NaN,NaN,675000.00,2250000.00,85518.000,1624797.00,0.0,0.000,0.0,0.0,58369.5,58369.5


### 2.2 Categorical features: main function

In [4]:
def agg_categorical_features(df, id_col, categorical_cols, prefix):
        temp = df[[id_col] + categorical_cols].copy()
        temp[categorical_cols] = (temp[categorical_cols].fillna("Missing"))
        dummies = pd.get_dummies(temp[categorical_cols], prefix = [prefix + "_" + col for col in categorical_cols], dtype = int)
        dummies[id_col] = temp[id_col]
        counts = (dummies.groupby(id_col).sum())
        shares = (dummies.groupby(id_col).mean())
        counts.columns = [col + "_COUNT" for col in counts.columns]
        shares.columns = [col + "_SHARE" for col in shares.columns]
        result = pd.concat([counts, shares], axis=1)

        return result

For example:

In [8]:
#categorical_cols_bureau_df = ["CREDIT_ACTIVE", "CREDIT_CURRENCY", "CREDIT_TYPE"]

In [11]:
#bureau_cat = make_categorical_features(bureau_df, "SK_ID_CURR", categorical_cols_bureau_df, "BUREAU")
#bureau_cat

,BUREAU_CREDIT_ACTIVE_Active_COUNT,BUREAU_CREDIT_ACTIVE_Bad debt_COUNT,BUREAU_CREDIT_ACTIVE_Closed_COUNT,BUREAU_CREDIT_ACTIVE_Sold_COUNT,BUREAU_CREDIT_CURRENCY_currency 1_COUNT,BUREAU_CREDIT_CURRENCY_currency 2_COUNT,BUREAU_CREDIT_CURRENCY_currency 3_COUNT,BUREAU_CREDIT_CURRENCY_currency 4_COUNT,BUREAU_CREDIT_TYPE_Another type of loan_COUNT,BUREAU_CREDIT_TYPE_Car loan_COUNT,...,BUREAU_CREDIT_TYPE_Interbank credit_SHARE,BUREAU_CREDIT_TYPE_Loan for business development_SHARE,BUREAU_CREDIT_TYPE_Loan for purchase of shares (margin lending)_SHARE,BUREAU_CREDIT_TYPE_Loan for the purchase of equipment_SHARE,BUREAU_CREDIT_TYPE_Loan for working capital replenishment_SHARE,BUREAU_CREDIT_TYPE_Microloan_SHARE,BUREAU_CREDIT_TYPE_Mobile operator loan_SHARE,BUREAU_CREDIT_TYPE_Mortgage_SHARE,BUREAU_CREDIT_TYPE_Real estate loan_SHARE,BUREAU_CREDIT_TYPE_Unknown type of loan_SHARE
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,
100001,3,0,4,0,7,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
100002,2,0,6,0,8,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
100003,1,0,3,0,4,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
100004,0,0,2,0,2,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
100005,2,0,1,0,3,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
456249,2,0,11,0,13,0,0,0,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
456250,2,0,1,0,3,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
456253,2,0,2,0,4,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 2.3 Main function for tables to client:

In [6]:
def add_contract_feat_to_client(df, prefix):
    sum_cols = [col for col in df.columns if col.endswith("_COUNT") or col.endswith("_SUM")]
    other_cols = [col for col in df.columns if col not in ["SK_ID_PREV", "SK_ID_CURR"] and col not in sum_cols]
    tables = []

    if len(sum_cols) > 0:
        client_sum = (df.groupby("SK_ID_CURR")[sum_cols].sum())
        tables.append(client_sum)

    if len(other_cols) > 0:
        client_other = (df.groupby("SK_ID_CURR")[other_cols].agg(["median", "max"]))

        client_other.columns = [col + "_" + func for col, func in client_other.columns]
        tables.append(client_other)

    contract_count = (df.groupby("SK_ID_CURR")["SK_ID_PREV"].nunique().rename(prefix + "_CONTRACT_WITH_HISTORY_COUNT").to_frame())
    tables.append(contract_count)
    result = pd.concat(tables, axis=1)
    return result

#### 2.3 Bureau_df

In [7]:
bureau_df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 17 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   SK_ID_CURR              1716428 non-null  int64  
 1   SK_ID_BUREAU            1716428 non-null  int64  
 2   CREDIT_ACTIVE           1716428 non-null  object 
 3   CREDIT_CURRENCY         1716428 non-null  object 
 4   DAYS_CREDIT             1716428 non-null  int64  
 5   CREDIT_DAY_OVERDUE      1716428 non-null  int64  
 6   DAYS_CREDIT_ENDDATE     1610875 non-null  float64
 7   DAYS_ENDDATE_FACT       1082775 non-null  float64
 8   AMT_CREDIT_MAX_OVERDUE  591940 non-null   float64
 9   CNT_CREDIT_PROLONG      1716428 non-null  int64  
 10  AMT_CREDIT_SUM          1716415 non-null  float64
 11  AMT_CREDIT_SUM_DEBT     1458759 non-null  float64
 12  AMT_CREDIT_SUM_LIMIT    1124648 non-null  float64
 13  AMT_CREDIT_SUM_OVERDUE  1716428 non-null  float64
 14  CR

In [8]:
bureau_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("bureau.csv", case = False, na=False)]
display(bureau_description[["Row", "Description"]])

,Row,Description
122,SK_ID_CURR,ID of loan in our sample - one loan in our sam...
123,SK_BUREAU_ID,Recoded ID of previous Credit Bureau credit re...
124,CREDIT_ACTIVE,Status of the Credit Bureau (CB) reported credits
125,CREDIT_CURRENCY,Recoded currency of the Credit Bureau credit
126,DAYS_CREDIT,How many days before current application did c...
127,CREDIT_DAY_OVERDUE,Number of days past due on CB credit at the ti...
128,DAYS_CREDIT_ENDDATE,Remaining duration of CB credit (in days) at t...
129,DAYS_ENDDATE_FACT,Days since CB credit ended at the time of appl...
130,AMT_CREDIT_MAX_OVERDUE,Maximal amount overdue on the Credit Bureau cr...
131,CNT_CREDIT_PROLONG,How many times was the Credit Bureau credit pr...


Wi can split all features into numerical, categorical and special ones.

In [9]:
categorical_cols_bureau_df = ["CREDIT_ACTIVE", "CREDIT_CURRENCY", "CREDIT_TYPE"]
numerical_cols_bureau_df = ["AMT_CREDIT_MAX_OVERDUE", "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT","AMT_CREDIT_SUM_LIMIT", "AMT_CREDIT_SUM_OVERDUE", "AMT_ANNUITY"]
special_cols_bureau_df = ["SK_ID_CURR", "SK_ID_BUREAU", "DAYS_CREDIT", "DAYS_CREDIT_ENDDATE","DAYS_ENDDATE_FACT", "DAYS_CREDIT_UPDATE", "CREDIT_DAY_OVERDUE", "CNT_CREDIT_PROLONG"]

In [10]:
print("SPECIAL:", special_cols_bureau_df)
print("\nCATEGORICAL:", categorical_cols_bureau_df)
print("\nNUMERICAL:", numerical_cols_bureau_df)

SPECIAL: ['SK_ID_CURR', 'SK_ID_BUREAU', 'DAYS_CREDIT', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'DAYS_CREDIT_UPDATE', 'CREDIT_DAY_OVERDUE', 'CNT_CREDIT_PROLONG']

CATEGORICAL: ['CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'CREDIT_TYPE']

NUMERICAL: ['AMT_CREDIT_MAX_OVERDUE', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'AMT_ANNUITY']


### 2.2.1 Numerical

In [11]:
bureau_num = agg_numerical_features(bureau_df, "SK_ID_CURR", numerical_cols_bureau_df, "BUREAU")
bureau_num

,BUREAU_AMT_CREDIT_MAX_OVERDUE_median,BUREAU_AMT_CREDIT_MAX_OVERDUE_max,BUREAU_AMT_CREDIT_SUM_median,BUREAU_AMT_CREDIT_SUM_max,BUREAU_AMT_CREDIT_SUM_DEBT_median,BUREAU_AMT_CREDIT_SUM_DEBT_max,BUREAU_AMT_CREDIT_SUM_LIMIT_median,BUREAU_AMT_CREDIT_SUM_LIMIT_max,BUREAU_AMT_CREDIT_SUM_OVERDUE_median,BUREAU_AMT_CREDIT_SUM_OVERDUE_max,BUREAU_AMT_ANNUITY_median,BUREAU_AMT_ANNUITY_max
SK_ID_CURR,,,,,,,,,,,,
100001,NaN,NaN,168345.00,378000.00,0.000,373239.00,0.0,0.000,0.0,0.0,0.0,10822.5
100002,40.500,5043.645,54130.50,450000.00,0.000,245781.00,0.0,31988.565,0.0,0.0,0.0,0.0
100003,0.000,0.000,92576.25,810000.00,0.000,0.00,0.0,810000.000,0.0,0.0,NaN,NaN
100004,0.000,0.000,94518.90,94537.80,0.000,0.00,0.0,0.000,0.0,0.0,NaN,NaN
100005,0.000,0.000,58500.00,568800.00,25321.500,543087.00,0.0,0.000,0.0,0.0,0.0,4261.5
...,...,...,...,...,...,...,...,...,...,...,...,...
456249,0.000,18945.000,248692.50,765000.00,0.000,163071.00,0.0,0.000,0.0,0.0,NaN,NaN
456250,0.000,0.000,483349.50,2153110.05,391731.615,1840308.48,0.0,58268.385,0.0,0.0,51799.5,384147.0
456253,NaN,NaN,675000.00,2250000.00,85518.000,1624797.00,0.0,0.000,0.0,0.0,58369.5,58369.5


### 2.2.2 Categorical

In [12]:
bureau_cat = agg_categorical_features(bureau_df, "SK_ID_CURR", categorical_cols_bureau_df, "BUREAU")
bureau_cat

,BUREAU_CREDIT_ACTIVE_Active_COUNT,BUREAU_CREDIT_ACTIVE_Bad debt_COUNT,BUREAU_CREDIT_ACTIVE_Closed_COUNT,BUREAU_CREDIT_ACTIVE_Sold_COUNT,BUREAU_CREDIT_CURRENCY_currency 1_COUNT,BUREAU_CREDIT_CURRENCY_currency 2_COUNT,BUREAU_CREDIT_CURRENCY_currency 3_COUNT,BUREAU_CREDIT_CURRENCY_currency 4_COUNT,BUREAU_CREDIT_TYPE_Another type of loan_COUNT,BUREAU_CREDIT_TYPE_Car loan_COUNT,...,BUREAU_CREDIT_TYPE_Interbank credit_SHARE,BUREAU_CREDIT_TYPE_Loan for business development_SHARE,BUREAU_CREDIT_TYPE_Loan for purchase of shares (margin lending)_SHARE,BUREAU_CREDIT_TYPE_Loan for the purchase of equipment_SHARE,BUREAU_CREDIT_TYPE_Loan for working capital replenishment_SHARE,BUREAU_CREDIT_TYPE_Microloan_SHARE,BUREAU_CREDIT_TYPE_Mobile operator loan_SHARE,BUREAU_CREDIT_TYPE_Mortgage_SHARE,BUREAU_CREDIT_TYPE_Real estate loan_SHARE,BUREAU_CREDIT_TYPE_Unknown type of loan_SHARE
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,
100001,3,0,4,0,7,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
100002,2,0,6,0,8,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
100003,1,0,3,0,4,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
100004,0,0,2,0,2,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
100005,2,0,1,0,3,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
456249,2,0,11,0,13,0,0,0,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
456250,2,0,1,0,3,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
456253,2,0,2,0,4,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 2.2.3 Special

We can calculate the count of credits:

In [13]:
bureau_credit_count = (bureau_df.groupby("SK_ID_CURR")["SK_ID_BUREAU"].count().rename("BUREAU_CREDIT_COUNT"))
bureau_credit_count.shape

(305811,)

In [ ]:
#bureau_df["CREDIT_ACTIVE"].value_counts(dropna=False)

And we can check whether there is active credit or not:

In [14]:
bureau_df["IS_ACTIVE_CREDIT"] = (bureau_df["CREDIT_ACTIVE"] == "Active").astype(int)
bureau_active = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_ACTIVE_COUNT = ("IS_ACTIVE_CREDIT", "sum"), BUREAU_ACTIVE_SHARE = ("IS_ACTIVE_CREDIT", "mean")))
bureau_active.head()

,BUREAU_ACTIVE_COUNT,BUREAU_ACTIVE_SHARE
SK_ID_CURR,,
100001,3,0.428571
100002,2,0.250000
100003,1,0.250000
100004,0,0.000000
100005,2,0.666667


We can check how many days of credit are available:

In [15]:
bureau_df["DAYS_CREDIT"].describe()

count    1.716428e+06
mean    -1.142108e+03
std      7.951649e+02
min     -2.922000e+03
25%     -1.666000e+03
50%     -9.870000e+02
75%     -4.740000e+02
max      0.000000e+00
Name: DAYS_CREDIT, dtype: float64


max indicates how recently the last loan was taken out,
min indicates how far back the credit history goes,
median typical age of the client's loans

In [16]:
bureau_days_credit = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_MOST_RECENT_CREDIT_DAYS = ("DAYS_CREDIT", "max"), BUREAU_OLDEST_CREDIT_DAYS = ("DAYS_CREDIT", "min"), BUREAU_MEDIAN_CREDIT_DAYS = ("DAYS_CREDIT", "median")))
bureau_days_credit.head()

,BUREAU_MOST_RECENT_CREDIT_DAYS,BUREAU_OLDEST_CREDIT_DAYS,BUREAU_MEDIAN_CREDIT_DAYS
SK_ID_CURR,,,
100001,-49,-1572,-857.0
100002,-103,-1437,-1042.5
100003,-606,-2586,-1205.5
100004,-408,-1326,-867.0
100005,-62,-373,-137.0


We can obtain the length of the client's credit history:

In [17]:
bureau_days_credit["BUREAU_CREDIT_HISTORY_LENGTH"] = (bureau_days_credit["BUREAU_MOST_RECENT_CREDIT_DAYS"] - bureau_days_credit["BUREAU_OLDEST_CREDIT_DAYS"])
bureau_days_credit

,BUREAU_MOST_RECENT_CREDIT_DAYS,BUREAU_OLDEST_CREDIT_DAYS,BUREAU_MEDIAN_CREDIT_DAYS,BUREAU_CREDIT_HISTORY_LENGTH
SK_ID_CURR,,,,
100001,-49,-1572,-857.0,1523
100002,-103,-1437,-1042.5,1334
100003,-606,-2586,-1205.5,1980
100004,-408,-1326,-867.0,918
100005,-62,-373,-137.0,311
...,...,...,...,...
456249,-483,-2713,-1680.0,2230
456250,-760,-1002,-824.0,242
456253,-713,-919,-919.0,206


In [ ]:
#bureau_df["CREDIT_DAY_OVERDUE"].describe()

On this part we look at overdue:

In [18]:
bureau_df["IS_DAY_OVERDUE"] = (bureau_df["CREDIT_DAY_OVERDUE"] > 0).astype(int)

In [19]:
bureau_overdue = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_MAX_DAYS_OVERDUE=("CREDIT_DAY_OVERDUE", "max"), BUREAU_OVERDUE_CREDIT_COUNT=("IS_DAY_OVERDUE", "sum"), BUREAU_OVERDUE_CREDIT_SHARE=("IS_DAY_OVERDUE", "mean")))
bureau_overdue.head(20)

,BUREAU_MAX_DAYS_OVERDUE,BUREAU_OVERDUE_CREDIT_COUNT,BUREAU_OVERDUE_CREDIT_SHARE
SK_ID_CURR,,,
100001,0,0,0.0
100002,0,0,0.0
100003,0,0,0.0
100004,0,0,0.0
100005,0,0,0.0
100007,0,0,0.0
100008,0,0,0.0
100009,0,0,0.0
100010,0,0,0.0


In [20]:
print("Доля записей с просрочкой:", (bureau_df["CREDIT_DAY_OVERDUE"] > 0).mean())
print("Количество записей с просрочкой:", (bureau_df["CREDIT_DAY_OVERDUE"] > 0).sum())

Доля записей с просрочкой: 0.0024568464275809996
Количество записей с просрочкой: 4217


In [ ]:
#(bureau_overdue["BUREAU_OVERDUE_CREDIT_COUNT"] > 0).mean()

In [ ]:
# temp = application_train_df[["SK_ID_CURR", "TARGET"]].merge(bureau_overdue, on="SK_ID_CURR", how="left")
# temp["HAS_BUREAU_OVERDUE"] = (temp["BUREAU_OVERDUE_CREDIT_COUNT"].fillna(0) > 0).astype(int)
# temp.groupby("HAS_BUREAU_OVERDUE")["TARGET"].agg(["count", "mean"])

We will collect data on the loan extension:

In [21]:
bureau_df["IS_PROLONGED"] = (bureau_df["CNT_CREDIT_PROLONG"] > 0).astype(int)
bureau_prolong = bureau_df.groupby("SK_ID_CURR").agg(BUREAU_TOTAL_PROLONG = ("CNT_CREDIT_PROLONG", "sum"), BUREAU_MAX_PROLONG = ("CNT_CREDIT_PROLONG", "max"), BUREAU_PROLONG_SHARE = ("IS_PROLONGED", "mean"))
bureau_prolong

,BUREAU_TOTAL_PROLONG,BUREAU_MAX_PROLONG,BUREAU_PROLONG_SHARE
SK_ID_CURR,,,
100001,0,0,0.000000
100002,0,0,0.000000
100003,0,0,0.000000
100004,0,0,0.000000
100005,0,0,0.000000
...,...,...,...
456249,0,0,0.000000
456250,0,0,0.000000
456253,0,0,0.000000


We will collect data of the days credit end data:

In [22]:
bureau_df["ENDS_AFTER_APPLICATION"] = np.where(bureau_df["DAYS_CREDIT_ENDDATE"].isna(), np.nan, (bureau_df["DAYS_CREDIT_ENDDATE"] > 0).astype(int))
bureau_credit_enddate = bureau_df.groupby("SK_ID_CURR").agg(BUREAU_LAST_PLANNED_ENDDATE=("DAYS_CREDIT_ENDDATE", "max"), BUREAU_MEAN_PLANNED_ENDDATE=("DAYS_CREDIT_ENDDATE", "mean"), BUREAU_ENDS_AFTER_APPL_SHARE=("ENDS_AFTER_APPLICATION", "mean"))
bureau_credit_enddate

,BUREAU_LAST_PLANNED_ENDDATE,BUREAU_MEAN_PLANNED_ENDDATE,BUREAU_ENDS_AFTER_APPL_SHARE
SK_ID_CURR,,,
100001,1778.0,82.428571,0.428571
100002,780.0,-349.000000,0.500000
100003,1216.0,-544.500000,0.250000
100004,-382.0,-488.500000,0.000000
100005,1324.0,439.333333,0.666667
...,...,...,...
456249,1363.0,-1232.333333,0.083333
456250,2340.0,1288.333333,0.666667
456253,1113.0,280.500000,0.500000


In [23]:
bureau_fact_enddate = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_LAST_FACT_ENDDATE = ("DAYS_ENDDATE_FACT", "max"), BUREAU_MEAN_FACT_ENDDATE = ("DAYS_ENDDATE_FACT", "mean")))
bureau_fact_enddate

,BUREAU_LAST_FACT_ENDDATE,BUREAU_MEAN_FACT_ENDDATE
SK_ID_CURR,,
100001,-544.0,-825.500000
100002,-36.0,-697.500000
100003,-540.0,-1097.333333
100004,-382.0,-532.500000
100005,-123.0,-123.000000
...,...,...
456249,-291.0,-1364.750000
456250,-760.0,-760.000000
456253,-794.0,-794.000000


And we will find the difference between the planned (fact) and actual dates:

In [24]:
bureau_df["ENDDATE_DIFF"] = bureau_df["DAYS_ENDDATE_FACT"] - bureau_df["DAYS_CREDIT_ENDDATE"]
bureau_enddate_diff = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_MEDIAN_DIFF_ENDDATE = ("ENDDATE_DIFF", "median"), BUREAU_MAX_DIFF_ENDDATE = ("ENDDATE_DIFF", "max")))
bureau_enddate_diff

,BUREAU_MEDIAN_DIFF_ENDDATE,BUREAU_MAX_DIFF_ENDDATE
SK_ID_CURR,,
100001,-45.5,1.0
100002,-113.0,0.0
100003,0.0,303.0
100004,-44.0,0.0
100005,5.0,5.0
...,...,...
456249,0.0,179.0
456250,-488.0,-488.0
456253,-605.0,-605.0


In [25]:
bureau_update = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_LAST_UPDATE_DAYS = ("DAYS_CREDIT_UPDATE", "max"), BUREAU_MEAN_UPDATE_DAYS = ("DAYS_CREDIT_UPDATE", "mean")))
bureau_update

,BUREAU_LAST_UPDATE_DAYS,BUREAU_MEAN_UPDATE_DAYS
SK_ID_CURR,,
100001,-6,-93.142857
100002,-7,-499.875000
100003,-43,-816.000000
100004,-382,-532.000000
100005,-11,-54.333333
...,...,...
456249,-12,-1064.538462
456250,-23,-60.333333
456253,-5,-253.250000


We now synthesize the ratio of monetary quantities to others:

ratio1 = current debt / original loan amount

In [26]:
bureau_df["DEBT_TO_CREDIT"] = (bureau_df["AMT_CREDIT_SUM_DEBT"] / bureau_df["AMT_CREDIT_SUM"].replace(0, np.nan))

ratio2 = overdue loan amount / loan amount

In [27]:
bureau_df["OVERDUE_TO_CREDIT"] = bureau_df["AMT_CREDIT_SUM_OVERDUE"] / bureau_df["AMT_CREDIT_SUM"].replace(0, np.nan)

In [28]:
bureau_ratios = bureau_df.groupby("SK_ID_CURR").agg(BUREAU_MEDIAN_DEBT_RATIO = ("DEBT_TO_CREDIT", "median"), BUREAU_MAX_DEBT_RATIO = ("DEBT_TO_CREDIT", "max"), BUREAU_MEDIAN_OVERDUE_RATIO = ("OVERDUE_TO_CREDIT", "median"), BUREAU_MAX_OVERDUE_RATIO = ("OVERDUE_TO_CREDIT", "max"))
bureau_ratios

,BUREAU_MEDIAN_DEBT_RATIO,BUREAU_MAX_DEBT_RATIO,BUREAU_MEDIAN_OVERDUE_RATIO,BUREAU_MAX_OVERDUE_RATIO
SK_ID_CURR,,,,
100001,0.000000,0.987405,0.0,0.0
100002,0.000000,0.546180,0.0,0.0
100003,0.000000,0.000000,0.0,0.0
100004,0.000000,0.000000,0.0,0.0
100005,0.848974,0.954794,0.0,0.0
...,...,...,...,...
456249,0.000000,0.905950,0.0,0.0
456250,0.854721,0.870515,0.0,0.0
456253,0.237550,0.722132,0.0,0.0


Finally, we will gather all the special features:

In [29]:
bureau_special = pd.concat(
    [
        bureau_credit_count,
        bureau_active,
        bureau_days_credit,
        bureau_overdue,
        bureau_prolong,
        bureau_credit_enddate,
        bureau_fact_enddate,
        bureau_enddate_diff,
        bureau_update,
        bureau_ratios
    ],
    axis=1
)
bureau_special.head()

,BUREAU_CREDIT_COUNT,BUREAU_ACTIVE_COUNT,BUREAU_ACTIVE_SHARE,BUREAU_MOST_RECENT_CREDIT_DAYS,BUREAU_OLDEST_CREDIT_DAYS,BUREAU_MEDIAN_CREDIT_DAYS,BUREAU_CREDIT_HISTORY_LENGTH,BUREAU_MAX_DAYS_OVERDUE,BUREAU_OVERDUE_CREDIT_COUNT,BUREAU_OVERDUE_CREDIT_SHARE,...,BUREAU_LAST_FACT_ENDDATE,BUREAU_MEAN_FACT_ENDDATE,BUREAU_MEDIAN_DIFF_ENDDATE,BUREAU_MAX_DIFF_ENDDATE,BUREAU_LAST_UPDATE_DAYS,BUREAU_MEAN_UPDATE_DAYS,BUREAU_MEDIAN_DEBT_RATIO,BUREAU_MAX_DEBT_RATIO,BUREAU_MEDIAN_OVERDUE_RATIO,BUREAU_MAX_OVERDUE_RATIO
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,
100001,7,3,0.428571,-49,-1572,-857.0,1523,0,0,0.0,...,-544.0,-825.500000,-45.5,1.0,-6,-93.142857,0.000000,0.987405,0.0,0.0
100002,8,2,0.250000,-103,-1437,-1042.5,1334,0,0,0.0,...,-36.0,-697.500000,-113.0,0.0,-7,-499.875000,0.000000,0.546180,0.0,0.0
100003,4,1,0.250000,-606,-2586,-1205.5,1980,0,0,0.0,...,-540.0,-1097.333333,0.0,303.0,-43,-816.000000,0.000000,0.000000,0.0,0.0
100004,2,0,0.000000,-408,-1326,-867.0,918,0,0,0.0,...,-382.0,-532.500000,-44.0,0.0,-382,-532.000000,0.000000,0.000000,0.0,0.0
100005,3,2,0.666667,-62,-373,-137.0,311,0,0,0.0,...,-123.0,-123.000000,5.0,5.0,-11,-54.333333,0.848974,0.954794,0.0,0.0


All features for bureau_df:

In [30]:
bureau_features = pd.concat([bureau_num, bureau_cat, bureau_special], axis=1)
bureau_features.head()

,BUREAU_AMT_CREDIT_MAX_OVERDUE_median,BUREAU_AMT_CREDIT_MAX_OVERDUE_max,BUREAU_AMT_CREDIT_SUM_median,BUREAU_AMT_CREDIT_SUM_max,BUREAU_AMT_CREDIT_SUM_DEBT_median,BUREAU_AMT_CREDIT_SUM_DEBT_max,BUREAU_AMT_CREDIT_SUM_LIMIT_median,BUREAU_AMT_CREDIT_SUM_LIMIT_max,BUREAU_AMT_CREDIT_SUM_OVERDUE_median,BUREAU_AMT_CREDIT_SUM_OVERDUE_max,...,BUREAU_LAST_FACT_ENDDATE,BUREAU_MEAN_FACT_ENDDATE,BUREAU_MEDIAN_DIFF_ENDDATE,BUREAU_MAX_DIFF_ENDDATE,BUREAU_LAST_UPDATE_DAYS,BUREAU_MEAN_UPDATE_DAYS,BUREAU_MEDIAN_DEBT_RATIO,BUREAU_MAX_DEBT_RATIO,BUREAU_MEDIAN_OVERDUE_RATIO,BUREAU_MAX_OVERDUE_RATIO
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,
100001,NaN,NaN,168345.00,378000.0,0.0,373239.0,0.0,0.000,0.0,0.0,...,-544.0,-825.500000,-45.5,1.0,-6,-93.142857,0.000000,0.987405,0.0,0.0
100002,40.5,5043.645,54130.50,450000.0,0.0,245781.0,0.0,31988.565,0.0,0.0,...,-36.0,-697.500000,-113.0,0.0,-7,-499.875000,0.000000,0.546180,0.0,0.0
100003,0.0,0.000,92576.25,810000.0,0.0,0.0,0.0,810000.000,0.0,0.0,...,-540.0,-1097.333333,0.0,303.0,-43,-816.000000,0.000000,0.000000,0.0,0.0
100004,0.0,0.000,94518.90,94537.8,0.0,0.0,0.0,0.000,0.0,0.0,...,-382.0,-532.500000,-44.0,0.0,-382,-532.000000,0.000000,0.000000,0.0,0.0
100005,0.0,0.000,58500.00,568800.0,25321.5,543087.0,0.0,0.000,0.0,0.0,...,-123.0,-123.000000,5.0,5.0,-11,-54.333333,0.848974,0.954794,0.0,0.0


In [31]:
print("Размер:", bureau_features.shape)
print("Уникальный SK_ID_CURR:", bureau_features.index.is_unique)
print("Количество клиентов:", bureau_features.shape[0])

Размер: (305811, 84)
Уникальный SK_ID_CURR: True
Количество клиентов: 305811


#### 2.4 Bureau_balance_df

In [ ]:
#bureau_balance_df = pd.read_csv(path_to_data+"bureau_balance.csv")

In [32]:
bureau_balance_df.info(show_counts=True)
bureau_balance_df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27299925 entries, 0 to 27299924
Data columns (total 3 columns):
 #   Column          Non-Null Count     Dtype 
---  ------          --------------     ----- 
 0   SK_ID_BUREAU    27299925 non-null  int64 
 1   MONTHS_BALANCE  27299925 non-null  int64 
 2   STATUS          27299925 non-null  object
dtypes: int64(2), object(1)
memory usage: 624.8+ MB


(27299925, 3)

In [33]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
bureau_balance_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("bureau_balance.csv", case = False, na=False)]
display(bureau_balance_description[["Row", "Description"]])

,Row,Description
139,SK_BUREAU_ID,Recoded ID of Credit Bureau credit (unique cod...
140,MONTHS_BALANCE,Month of balance relative to application date ...
141,STATUS,Status of Credit Bureau loan during the month ...


In [34]:
categorical_cols_bureau_balance_df = ["STATUS"]
numerical_cols_bureau_balance_df = []
special_cols_bureau_balance_df = ["SK_ID_BUREAU", "MONTHS_BALANCE"]

In [35]:
print("SPECIAL:", special_cols_bureau_balance_df)
print("\nCATEGORICAL:", categorical_cols_bureau_balance_df)
print("\nNUMERICAL:", numerical_cols_bureau_balance_df)

SPECIAL: ['SK_ID_BUREAU', 'MONTHS_BALANCE']

CATEGORICAL: ['STATUS']

NUMERICAL: []


### 2.4.1 Categorical

In [36]:
bureau_balance_df["STATUS"].value_counts(dropna=False)

STATUS
C    13646993
0     7499507
X     5810482
1      242347
5       62406
2       23419
3        8924
4        5847
Name: count, dtype: int64

In [37]:
bureau_balance_df["IS_BB_OVERDUE"] = bureau_balance_df["STATUS"].isin(["1", "2", "3", "4", "5"]).astype(int)

In [38]:
bureau_balance_df["IS_BB_OVERDUE"].value_counts()

IS_BB_OVERDUE
0    26956982
1      342943
Name: count, dtype: int64

In [39]:
bureau_balance_df["IS_BB_OVERDUE"].mean()

np.float64(0.012562049163138727)

We can add hard overdue:

In [40]:
bureau_balance_df["IS_BB_SEVERE_OVERDUE"] = bureau_balance_df["STATUS"].isin(["3", "4", "5"]).astype(int)

In [41]:
bureau_balance_cat = agg_categorical_features(bureau_balance_df, "SK_ID_BUREAU", categorical_cols_bureau_balance_df, "BB")
bureau_balance_cat

,BB_STATUS_0_COUNT,BB_STATUS_1_COUNT,BB_STATUS_2_COUNT,BB_STATUS_3_COUNT,BB_STATUS_4_COUNT,BB_STATUS_5_COUNT,BB_STATUS_C_COUNT,BB_STATUS_X_COUNT,BB_STATUS_0_SHARE,BB_STATUS_1_SHARE,BB_STATUS_2_SHARE,BB_STATUS_3_SHARE,BB_STATUS_4_SHARE,BB_STATUS_5_SHARE,BB_STATUS_C_SHARE,BB_STATUS_X_SHARE
SK_ID_BUREAU,,,,,,,,,,,,,,,,
5001709,0,0,0,0,0,0,86,11,0.000000,0.000000,0.0,0.0,0.0,0.0,0.886598,0.113402
5001710,5,0,0,0,0,0,48,30,0.060241,0.000000,0.0,0.0,0.0,0.0,0.578313,0.361446
5001711,3,0,0,0,0,0,0,1,0.750000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.250000
5001712,10,0,0,0,0,0,9,0,0.526316,0.000000,0.0,0.0,0.0,0.0,0.473684,0.000000
5001713,0,0,0,0,0,0,0,22,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6842884,9,0,0,0,0,0,20,19,0.187500,0.000000,0.0,0.0,0.0,0.0,0.416667,0.395833
6842885,12,0,0,0,0,12,0,0,0.500000,0.000000,0.0,0.0,0.0,0.5,0.000000,0.000000
6842886,8,0,0,0,0,0,25,0,0.242424,0.000000,0.0,0.0,0.0,0.0,0.757576,0.000000


### 2.4.2 Special

In [42]:
bureau_balance_df["MONTHS_BALANCE"].value_counts(dropna=False)

MONTHS_BALANCE
-1     622601
-2     619243
-3     615080
 0     610965
-4     609138
        ...  
-92     57300
-93     53535
-94     49965
-95     46542
-96     43147
Name: count, Length: 97, dtype: int64

In [43]:
bureau_balance_special = (bureau_balance_df.groupby("SK_ID_BUREAU").agg(BB_MONTH_COUNT=("MONTHS_BALANCE","count"),
BB_OLDEST_MONTH=("MONTHS_BALANCE","min"), BB_MOST_RECENT_MONTH=("MONTHS_BALANCE", "max"), BB_OVERDUE_MONTH_COUNT=("IS_BB_OVERDUE", "sum"), BB_SEVERE_OVERDUE_MONTH_COUNT=("IS_BB_SEVERE_OVERDUE", "sum")))
bureau_balance_special

,BB_MONTH_COUNT,BB_OLDEST_MONTH,BB_MOST_RECENT_MONTH,BB_OVERDUE_MONTH_COUNT,BB_SEVERE_OVERDUE_MONTH_COUNT
SK_ID_BUREAU,,,,,
5001709,97,-96,0,0,0
5001710,83,-82,0,0,0
5001711,4,-3,0,0,0
5001712,19,-18,0,0,0
5001713,22,-21,0,0,0
...,...,...,...,...,...
6842884,48,-47,0,0,0
6842885,24,-23,0,12,12
6842886,33,-32,0,0,0


In [44]:
bureau_balance_special["BB_HISTORY_LENGTH"] = (bureau_balance_special["BB_MOST_RECENT_MONTH"] - bureau_balance_special["BB_OLDEST_MONTH"])

In [45]:
bureau_balance_features = pd.concat([bureau_balance_cat, bureau_balance_special], axis=1)
bureau_balance_features.head()

,BB_STATUS_0_COUNT,BB_STATUS_1_COUNT,BB_STATUS_2_COUNT,BB_STATUS_3_COUNT,BB_STATUS_4_COUNT,BB_STATUS_5_COUNT,BB_STATUS_C_COUNT,BB_STATUS_X_COUNT,BB_STATUS_0_SHARE,BB_STATUS_1_SHARE,...,BB_STATUS_4_SHARE,BB_STATUS_5_SHARE,BB_STATUS_C_SHARE,BB_STATUS_X_SHARE,BB_MONTH_COUNT,BB_OLDEST_MONTH,BB_MOST_RECENT_MONTH,BB_OVERDUE_MONTH_COUNT,BB_SEVERE_OVERDUE_MONTH_COUNT,BB_HISTORY_LENGTH
SK_ID_BUREAU,,,,,,,,,,,,,,,,,,,,,
5001709,0,0,0,0,0,0,86,11,0.000000,0.0,...,0.0,0.0,0.886598,0.113402,97,-96,0,0,0,96
5001710,5,0,0,0,0,0,48,30,0.060241,0.0,...,0.0,0.0,0.578313,0.361446,83,-82,0,0,0,82
5001711,3,0,0,0,0,0,0,1,0.750000,0.0,...,0.0,0.0,0.000000,0.250000,4,-3,0,0,0,3
5001712,10,0,0,0,0,0,9,0,0.526316,0.0,...,0.0,0.0,0.473684,0.000000,19,-18,0,0,0,18
5001713,0,0,0,0,0,0,0,22,0.000000,0.0,...,0.0,0.0,0.000000,1.000000,22,-21,0,0,0,21


In [46]:
print("Уникальный SK_ID_BUREAU:", bureau_balance_features.index.is_unique)
print("Размер:", bureau_balance_features.shape)

Уникальный SK_ID_BUREAU: True
Размер: (817395, 22)


We will merge 3 tables: bureau_df and bureau_balance_df to application_train_df

In [47]:
 bureau_balance_with_client = (bureau_balance_features.reset_index().merge(bureau_df[["SK_ID_BUREAU", "SK_ID_CURR"]], on = "SK_ID_BUREAU", how = "left", validate = "1:1"))
 bureau_balance_with_client

,SK_ID_BUREAU,BB_STATUS_0_COUNT,BB_STATUS_1_COUNT,BB_STATUS_2_COUNT,BB_STATUS_3_COUNT,BB_STATUS_4_COUNT,BB_STATUS_5_COUNT,BB_STATUS_C_COUNT,BB_STATUS_X_COUNT,BB_STATUS_0_SHARE,...,BB_STATUS_5_SHARE,BB_STATUS_C_SHARE,BB_STATUS_X_SHARE,BB_MONTH_COUNT,BB_OLDEST_MONTH,BB_MOST_RECENT_MONTH,BB_OVERDUE_MONTH_COUNT,BB_SEVERE_OVERDUE_MONTH_COUNT,BB_HISTORY_LENGTH,SK_ID_CURR
0,5001709,0,0,0,0,0,0,86,11,0.000000,...,0.0,0.886598,0.113402,97,-96,0,0,0,96,NaN
1,5001710,5,0,0,0,0,0,48,30,0.060241,...,0.0,0.578313,0.361446,83,-82,0,0,0,82,162368.0
2,5001711,3,0,0,0,0,0,0,1,0.750000,...,0.0,0.000000,0.250000,4,-3,0,0,0,3,162368.0
3,5001712,10,0,0,0,0,0,9,0,0.526316,...,0.0,0.473684,0.000000,19,-18,0,0,0,18,162368.0
4,5001713,0,0,0,0,0,0,0,22,0.000000,...,0.0,0.000000,1.000000,22,-21,0,0,0,21,150635.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
817390,6842884,9,0,0,0,0,0,20,19,0.187500,...,0.0,0.416667,0.395833,48,-47,0,0,0,47,387020.0
817391,6842885,12,0,0,0,0,12,0,0,0.500000,...,0.5,0.000000,0.000000,24,-23,0,12,12,23,387020.0
817392,6842886,8,0,0,0,0,0,25,0,0.242424,...,0.0,0.757576,0.000000,33,-32,0,0,0,32,387020.0
817393,6842887,6,0,0,0,0,0,31,0,0.162162,...,0.0,0.837838,0.000000,37,-36,0,0,0,36,387020.0


In [48]:
print("Кредитов без SK_ID_CURR:", bureau_balance_with_client["SK_ID_CURR"].isna().sum())

Кредитов без SK_ID_CURR: 43041


In [50]:
bb_ids = set(bureau_balance_features.index)
bureau_ids = set(bureau_df["SK_ID_BUREAU"])
unmapped_ids = (bb_ids - bureau_ids)

print("SK_ID_BUREAU в bureau_balance:", len(bb_ids))
print("SK_ID_BUREAU в bureau:", len(bureau_ids))
print("Есть в bureau_balance, но нет в bureau:", len(unmapped_ids))

SK_ID_BUREAU в bureau_balance: 817395
SK_ID_BUREAU в bureau: 1716428
Есть в bureau_balance, но нет в bureau: 43041


In [51]:
bureau_balance_mapped = (bureau_balance_with_client[bureau_balance_with_client["SK_ID_CURR"].notna()].copy())
bureau_balance_unmapped = (bureau_balance_with_client[bureau_balance_with_client["SK_ID_CURR"].isna()].copy())

In [53]:
bureau_balance_mapped["SK_ID_CURR"] = (bureau_balance_mapped["SK_ID_CURR"].astype("int64"))

Use "count" for 1 client:

In [55]:
bureau_balance_count_cols = [col for col in bureau_balance_mapped.columns if col.endswith("_COUNT")]
bureau_balance_count_cols

['BB_STATUS_0_COUNT',
 'BB_STATUS_1_COUNT',
 'BB_STATUS_2_COUNT',
 'BB_STATUS_3_COUNT',
 'BB_STATUS_4_COUNT',
 'BB_STATUS_5_COUNT',
 'BB_STATUS_C_COUNT',
 'BB_STATUS_X_COUNT',
 'BB_MONTH_COUNT',
 'BB_OVERDUE_MONTH_COUNT',
 'BB_SEVERE_OVERDUE_MONTH_COUNT']

In [56]:
bureau_balance_client_counts = (bureau_balance_mapped.groupby("SK_ID_CURR")[bureau_balance_count_cols].sum())
bureau_balance_client_counts

,BB_STATUS_0_COUNT,BB_STATUS_1_COUNT,BB_STATUS_2_COUNT,BB_STATUS_3_COUNT,BB_STATUS_4_COUNT,BB_STATUS_5_COUNT,BB_STATUS_C_COUNT,BB_STATUS_X_COUNT,BB_MONTH_COUNT,BB_OVERDUE_MONTH_COUNT,BB_SEVERE_OVERDUE_MONTH_COUNT
SK_ID_CURR,,,,,,,,,,,
100001,31,1,0,0,0,0,110,30,172,1,0
100002,45,27,0,0,0,0,23,15,110,27,0
100005,14,0,0,0,0,0,5,2,21,0,0
100010,20,0,0,0,0,0,52,0,72,0,0
100013,79,7,0,0,0,0,103,41,230,7,0
...,...,...,...,...,...,...,...,...,...,...,...
456247,66,0,0,0,0,0,219,35,320,0,0
456250,12,0,0,0,0,0,25,50,87,0,0
456253,47,0,0,0,0,0,57,13,117,0,0


We will add the remaining customer attributes:

In [57]:
bureau_balance_client_special = (bureau_balance_mapped.groupby("SK_ID_CURR").agg(BB_CREDIT_WITH_HISTORY_COUNT = ("SK_ID_BUREAU", "nunique"), BB_HISTORY_LENGTH_MAX = ("BB_HISTORY_LENGTH", "max"), BB_HISTORY_LENGTH_MEDIAN = ("BB_HISTORY_LENGTH", "median"), BB_OLDEST_MONTH_CLIENT = ("BB_OLDEST_MONTH", "min"), BB_MOST_RECENT_MONTH_CLIENT = ("BB_MOST_RECENT_MONTH", "max")))
bureau_balance_client_special

,BB_CREDIT_WITH_HISTORY_COUNT,BB_HISTORY_LENGTH_MAX,BB_HISTORY_LENGTH_MEDIAN,BB_OLDEST_MONTH_CLIENT,BB_MOST_RECENT_MONTH_CLIENT
SK_ID_CURR,,,,,
100001,7,51,28.0,-51,0
100002,8,21,15.0,-47,0
100005,3,12,4.0,-12,0
100010,2,35,35.0,-90,-2
100013,4,68,59.5,-68,0
...,...,...,...,...,...
456247,11,81,25.0,-81,0
456250,3,32,27.0,-32,0
456253,4,30,30.0,-30,0


We calculate the shares of overdue payments for each client:

In [63]:
bureau_balance_client_counts["BB_OVERDUE_MONTH_SHARE"] = (bureau_balance_client_counts["BB_OVERDUE_MONTH_COUNT"] / bureau_balance_client_counts["BB_MONTH_COUNT"].replace(0, np.nan))

bureau_balance_client_counts["BB_SEVERE_OVERDUE_MONTH_SHARE"] = (bureau_balance_client_counts["BB_SEVERE_OVERDUE_MONTH_COUNT"] / bureau_balance_client_counts["BB_MONTH_COUNT"].replace(0, np.nan))

In [64]:
bureau_balance_status_count_cols = [col for col in bureau_balance_client_counts.columns if col.startswith("BB_STATUS_")and col.endswith("_COUNT")]

In [65]:
for col in bureau_balance_status_count_cols:

    share_col = col.replace("_COUNT", "_SHARE")
    bureau_balance_client_counts[share_col] = (bureau_balance_client_counts[col] / bureau_balance_client_counts["BB_MONTH_COUNT"].replace(0, np.nan))

Add final tables:

In [66]:
bureau_balance_features_client = pd.concat([bureau_balance_client_counts, bureau_balance_client_special], axis=1)
bureau_balance_features_client.head()

,BB_STATUS_0_COUNT,BB_STATUS_1_COUNT,BB_STATUS_2_COUNT,BB_STATUS_3_COUNT,BB_STATUS_4_COUNT,BB_STATUS_5_COUNT,BB_STATUS_C_COUNT,BB_STATUS_X_COUNT,BB_MONTH_COUNT,BB_OVERDUE_MONTH_COUNT,...,BB_STATUS_3_SHARE,BB_STATUS_4_SHARE,BB_STATUS_5_SHARE,BB_STATUS_C_SHARE,BB_STATUS_X_SHARE,BB_CREDIT_WITH_HISTORY_COUNT,BB_HISTORY_LENGTH_MAX,BB_HISTORY_LENGTH_MEDIAN,BB_OLDEST_MONTH_CLIENT,BB_MOST_RECENT_MONTH_CLIENT
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,
100001,31,1,0,0,0,0,110,30,172,1,...,0.0,0.0,0.0,0.639535,0.174419,7,51,28.0,-51,0
100002,45,27,0,0,0,0,23,15,110,27,...,0.0,0.0,0.0,0.209091,0.136364,8,21,15.0,-47,0
100005,14,0,0,0,0,0,5,2,21,0,...,0.0,0.0,0.0,0.238095,0.095238,3,12,4.0,-12,0
100010,20,0,0,0,0,0,52,0,72,0,...,0.0,0.0,0.0,0.722222,0.000000,2,35,35.0,-90,-2
100013,79,7,0,0,0,0,103,41,230,7,...,0.0,0.0,0.0,0.447826,0.178261,4,68,59.5,-68,0


In [68]:
print("Размер:", bureau_balance_features_client.shape)
print("SK_ID_CURR уникален:", bureau_balance_features_client.index.is_unique)
print("Пропусков в ID:", bureau_balance_features_client.index.isna().sum())

Размер: (134542, 26)
SK_ID_CURR уникален: True
Пропусков в ID: 0


## On the class

In [ ]:
# transf_tables_bureau = []
#
# for feat in numerical_cols_bureau_df:
#     transf_bureau = bureau_df.groupby("SK_ID_CURR")[feat].agg(agg_func)
#     transf_bureau.columns = [feat+"_"+func for func in agg_func]
#     transf_tables_bureau.append(transf_bureau)
# bureau_transformed = pd.concat(transf_tables_bureau, axis=1)
# bureau_transformed


In [ ]:
# application_train_df = application_train_df.merge(bureau_transformed, on="SK_ID_CURR", how="left")
# application_test_df = application_test_df.merge(bureau_transformed, on="SK_ID_CURR", how="left")
# application_train_df

In [ ]:
# transf_tables_bureau = []
#
# for feat in categorical_cols_bureau_df:
#     transf_bureau = bureau_df.groupby(["SK_ID_CURR", feat])[["SK_ID_BUREAU"]].count()
#     transf_bureau.columns = [feat+"_count"]
#     transf_bureau = transf_bureau.reset_index()
#     transf_bureau = transf_bureau.set_index("SK_ID_CURR")
#     transf_tables_bureau.append(transf_bureau)
#
#     display(transf_bureau)
# #bureau_transformed = pd.concat(transf_tables_bureau, axis=1)
# #bureau_transformed


In [ ]:
# transf_tables_bureau = []
#
# for feat in categorical_cols_bureau_df:
#     transf_bureau = bureau_df.pivot_table(index="SK_ID_CURR", columns=feat, aggfunc="count", fill_value=0, values="SK_ID_BUREAU")
#     #transf_tables_bureau.append(transf_bureau)
#
#     display(transf_bureau)
# #bureau_transformed = pd.concat(transf_tables_bureau, axis=1)
# #bureau_transformed

In [ ]:
# bureau_df.groupby(["SK_ID_CURR", "CREDIT_ACTIVE"])[["SK_ID_BUREAU"]].count()

In [ ]:
# bureau_days_credit_mean = bureau_df.groupby("SK_ID_CURR")[["DAYS_CREDIT"]].mean()
# bureau_days_credit_mean

In [ ]:
# bureau_days_credit_median = bureau_df.groupby("SK_ID_CURR")[["DAYS_CREDIT"]].median()
# bureau_days_credit_median

In [ ]:
# bureau_days_credit_mean.merge(bureau_days_credit_median, on="SK_ID_CURR")

#### 2.5 previous_application_df

In [69]:
previous_application_df = pd.read_csv(path_to_data+"previous_application.csv")

In [70]:
previous_application_df.info(show_counts=True)
previous_application_df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1670214 entries, 0 to 1670213
Data columns (total 37 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   SK_ID_PREV                   1670214 non-null  int64  
 1   SK_ID_CURR                   1670214 non-null  int64  
 2   NAME_CONTRACT_TYPE           1670214 non-null  object 
 3   AMT_ANNUITY                  1297979 non-null  float64
 4   AMT_APPLICATION              1670214 non-null  float64
 5   AMT_CREDIT                   1670213 non-null  float64
 6   AMT_DOWN_PAYMENT             774370 non-null   float64
 7   AMT_GOODS_PRICE              1284699 non-null  float64
 8   WEEKDAY_APPR_PROCESS_START   1670214 non-null  object 
 9   HOUR_APPR_PROCESS_START      1670214 non-null  int64  
 10  FLAG_LAST_APPL_PER_CONTRACT  1670214 non-null  object 
 11  NFLAG_LAST_APPL_IN_DAY       1670214 non-null  int64  
 12  RATE_DOWN_PAYMENT            774370 non-nu

(1670214, 37)

In [71]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
previous_application_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("previous_application.csv", case = False, na=False)]
display(previous_application_description[["Row", "Description"]])

,Row,Description
173,SK_ID_PREV,ID of previous credit in Home credit related t...
174,SK_ID_CURR,ID of loan in our sample
175,NAME_CONTRACT_TYPE,"Contract product type (Cash loan, consumer loa..."
176,AMT_ANNUITY,Annuity of previous application
177,AMT_APPLICATION,For how much credit did client ask on the prev...
178,AMT_CREDIT,Final credit amount on the previous applicatio...
179,AMT_DOWN_PAYMENT,Down payment on the previous application
180,AMT_GOODS_PRICE,Goods price of good that client asked for (if ...
181,WEEKDAY_APPR_PROCESS_START,On which day of the week did the client apply ...
182,HOUR_APPR_PROCESS_START,Approximately at what day hour did the client ...


In [72]:
categorical_cols_prev_app_df = ["NAME_CONTRACT_TYPE", "WEEKDAY_APPR_PROCESS_START","FLAG_LAST_APPL_PER_CONTRACT", "NAME_CASH_LOAN_PURPOSE", "NAME_CONTRACT_STATUS","NAME_PAYMENT_TYPE", "CODE_REJECT_REASON", "NAME_TYPE_SUITE", "NAME_CLIENT_TYPE","NAME_GOODS_CATEGORY", "NAME_PORTFOLIO", "NAME_PRODUCT_TYPE", "CHANNEL_TYPE","NAME_SELLER_INDUSTRY", "NAME_YIELD_GROUP", "PRODUCT_COMBINATION"]
numerical_cols_prev_app_df = ["AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT","AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE", "RATE_DOWN_PAYMENT", "RATE_INTEREST_PRIMARY", "RATE_INTEREST_PRIVILEGED"]
special_cols_prev_app_df = ["SK_ID_PREV", "SK_ID_CURR", "HOUR_APPR_PROCESS_START", "NFLAG_LAST_APPL_IN_DAY", "CNT_PAYMENT", "DAYS_DECISION", "DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION", "NFLAG_INSURED_ON_APPROVAL", "SELLERPLACE_AREA"]

In [73]:
print("SPECIAL:", special_cols_prev_app_df)
print("\nCATEGORICAL:", categorical_cols_prev_app_df)
print("\nNUMERICAL:", numerical_cols_prev_app_df)

SPECIAL: ['SK_ID_PREV', 'SK_ID_CURR', 'HOUR_APPR_PROCESS_START', 'NFLAG_LAST_APPL_IN_DAY', 'CNT_PAYMENT', 'DAYS_DECISION', 'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION', 'NFLAG_INSURED_ON_APPROVAL', 'SELLERPLACE_AREA']

CATEGORICAL: ['NAME_CONTRACT_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'FLAG_LAST_APPL_PER_CONTRACT', 'NAME_CASH_LOAN_PURPOSE', 'NAME_CONTRACT_STATUS', 'NAME_PAYMENT_TYPE', 'CODE_REJECT_REASON', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE', 'NAME_SELLER_INDUSTRY', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION']

NUMERICAL: ['AMT_ANNUITY', 'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE', 'RATE_DOWN_PAYMENT', 'RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED']


In [74]:
prev_work = previous_application_df.copy()

In [75]:
prev_days_cols = ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION"]

prev_work[prev_days_cols] = (prev_work[prev_days_cols].replace(365243, np.nan))

In [84]:
prev_num = agg_numerical_features(prev_work, "SK_ID_CURR", numerical_cols_prev_app_df, "PREV")
prev_num.head()

,PREV_AMT_ANNUITY_median,PREV_AMT_ANNUITY_max,PREV_AMT_APPLICATION_median,PREV_AMT_APPLICATION_max,PREV_AMT_CREDIT_median,PREV_AMT_CREDIT_max,PREV_AMT_DOWN_PAYMENT_median,PREV_AMT_DOWN_PAYMENT_max,PREV_AMT_GOODS_PRICE_median,PREV_AMT_GOODS_PRICE_max,PREV_RATE_DOWN_PAYMENT_median,PREV_RATE_DOWN_PAYMENT_max,PREV_RATE_INTEREST_PRIMARY_median,PREV_RATE_INTEREST_PRIMARY_max,PREV_RATE_INTEREST_PRIVILEGED_median,PREV_RATE_INTEREST_PRIVILEGED_max
SK_ID_CURR,,,,,,,,,,,,,,,,
100001,3951.000,3951.000,24835.50,24835.5,23787.00,23787.0,2520.0,2520.0,24835.5,24835.5,0.104326,0.104326,NaN,NaN,NaN,NaN
100002,9251.775,9251.775,179055.00,179055.0,179055.00,179055.0,0.0,0.0,179055.0,179055.0,0.000000,0.000000,NaN,NaN,NaN,NaN
100003,64567.665,98356.995,337500.00,900000.0,348637.50,1035882.0,3442.5,6885.0,337500.0,900000.0,0.050030,0.100061,NaN,NaN,NaN,NaN
100004,5357.250,5357.250,24282.00,24282.0,20106.00,20106.0,4860.0,4860.0,24282.0,24282.0,0.212008,0.212008,NaN,NaN,NaN,NaN
100005,4813.200,4813.200,22308.75,44617.5,20076.75,40153.5,4464.0,4464.0,44617.5,44617.5,0.108964,0.108964,NaN,NaN,NaN,NaN


In [85]:
prev_cat = agg_categorical_features(prev_work, "SK_ID_CURR", categorical_cols_prev_app_df, "PREV")
prev_cat.head()

,PREV_NAME_CONTRACT_TYPE_Cash loans_COUNT,PREV_NAME_CONTRACT_TYPE_Consumer loans_COUNT,PREV_NAME_CONTRACT_TYPE_Revolving loans_COUNT,PREV_NAME_CONTRACT_TYPE_XNA_COUNT,PREV_WEEKDAY_APPR_PROCESS_START_FRIDAY_COUNT,PREV_WEEKDAY_APPR_PROCESS_START_MONDAY_COUNT,PREV_WEEKDAY_APPR_PROCESS_START_SATURDAY_COUNT,PREV_WEEKDAY_APPR_PROCESS_START_SUNDAY_COUNT,PREV_WEEKDAY_APPR_PROCESS_START_THURSDAY_COUNT,PREV_WEEKDAY_APPR_PROCESS_START_TUESDAY_COUNT,...,PREV_PRODUCT_COMBINATION_Cash X-Sell: middle_SHARE,PREV_PRODUCT_COMBINATION_Missing_SHARE,PREV_PRODUCT_COMBINATION_POS household with interest_SHARE,PREV_PRODUCT_COMBINATION_POS household without interest_SHARE,PREV_PRODUCT_COMBINATION_POS industry with interest_SHARE,PREV_PRODUCT_COMBINATION_POS industry without interest_SHARE,PREV_PRODUCT_COMBINATION_POS mobile with interest_SHARE,PREV_PRODUCT_COMBINATION_POS mobile without interest_SHARE,PREV_PRODUCT_COMBINATION_POS other with interest_SHARE,PREV_PRODUCT_COMBINATION_POS others without interest_SHARE
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,
100001,0,1,0,0,1,0,0,0,0,0,...,0.0,0.0,0.000000,0.0,0.000000,0.0,1.0,0.0,0.0,0.0
100002,0,1,0,0,0,0,1,0,0,0,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,1.0,0.0
100003,1,2,0,0,1,0,1,1,0,0,...,0.0,0.0,0.333333,0.0,0.333333,0.0,0.0,0.0,0.0,0.0
100004,0,1,0,0,1,0,0,0,0,0,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,1.0,0.0,0.0
100005,1,1,0,0,1,0,0,0,1,0,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.5,0.0,0.0,0.0


In [86]:
prev_work["CREDIT_TO_APPLICATION"] = (prev_work["AMT_CREDIT"] / prev_work["AMT_APPLICATION"].replace(0, np.nan))
prev_work["DOWN_PAYMENT_SHARE"] = (prev_work["AMT_DOWN_PAYMENT"] / prev_work["AMT_APPLICATION"].replace(0, np.nan))
prev_work["ESTIMATED_TOTAL_PAYMENT"] = (prev_work["AMT_ANNUITY"] * prev_work["CNT_PAYMENT"])
prev_work["TOTAL_PAYMENT_TO_CREDIT"] = (prev_work["ESTIMATED_TOTAL_PAYMENT"] / prev_work["AMT_CREDIT"].replace(0, np.nan))

In [79]:
prev_special = (
    prev_work
    .groupby("SK_ID_CURR")
    .agg(
        PREV_APPLICATION_COUNT=(
            "SK_ID_PREV",
            "nunique"
        ),

        PREV_MOST_RECENT_DECISION=(
            "DAYS_DECISION",
            "max"
        ),

        PREV_OLDEST_DECISION=(
            "DAYS_DECISION",
            "min"
        ),

        PREV_CNT_PAYMENT_MEDIAN=(
            "CNT_PAYMENT",
            "median"
        ),

        PREV_CNT_PAYMENT_MAX=(
            "CNT_PAYMENT",
            "max"
        ),

        PREV_INSURED_SHARE=(
            "NFLAG_INSURED_ON_APPROVAL",
            "mean"
        ),

        PREV_LAST_APPL_IN_DAY_SHARE=(
            "NFLAG_LAST_APPL_IN_DAY",
            "mean"
        ),

        PREV_CREDIT_TO_APPLICATION_MEDIAN=(
            "CREDIT_TO_APPLICATION",
            "median"
        ),

        PREV_CREDIT_TO_APPLICATION_MAX=(
            "CREDIT_TO_APPLICATION",
            "max"
        ),

        PREV_DOWN_PAYMENT_SHARE_MEDIAN=(
            "DOWN_PAYMENT_SHARE",
            "median"
        ),

        PREV_TOTAL_PAYMENT_TO_CREDIT_MEDIAN=(
            "TOTAL_PAYMENT_TO_CREDIT",
            "median"
        )
    )
)

prev_special.head()

,PREV_APPLICATION_COUNT,PREV_MOST_RECENT_DECISION,PREV_OLDEST_DECISION,PREV_CNT_PAYMENT_MEDIAN,PREV_CNT_PAYMENT_MAX,PREV_INSURED_SHARE,PREV_LAST_APPL_IN_DAY_SHARE,PREV_CREDIT_TO_APPLICATION_MEDIAN,PREV_CREDIT_TO_APPLICATION_MAX,PREV_DOWN_PAYMENT_SHARE_MEDIAN,PREV_TOTAL_PAYMENT_TO_CREDIT_MEDIAN
SK_ID_CURR,,,,,,,,,,,
100001,1,-1740,-1740,8.0,8.0,0.000000,1.0,0.957782,0.957782,0.101468,1.328793
100002,1,-606,-606,24.0,24.0,0.000000,1.0,1.000000,1.000000,0.000000,1.240080
100003,3,-746,-2341,12.0,12.0,0.666667,1.0,1.033000,1.150980,0.050029,1.139400
100004,1,-815,-815,4.0,4.0,0.000000,1.0,0.828021,0.828021,0.200148,1.065801
100005,2,-315,-757,12.0,12.0,0.000000,1.0,0.899950,0.899950,0.100050,1.438440


In [80]:
prev_features = pd.concat([prev_num, prev_cat, prev_special], axis=1)
prev_features.head()

,PREV_AMT_ANNUITY_median,PREV_AMT_ANNUITY_max,PREV_AMT_APPLICATION_median,PREV_AMT_APPLICATION_max,PREV_AMT_CREDIT_median,PREV_AMT_CREDIT_max,PREV_AMT_DOWN_PAYMENT_median,PREV_AMT_DOWN_PAYMENT_max,PREV_AMT_GOODS_PRICE_median,PREV_AMT_GOODS_PRICE_max,...,PREV_MOST_RECENT_DECISION,PREV_OLDEST_DECISION,PREV_CNT_PAYMENT_MEDIAN,PREV_CNT_PAYMENT_MAX,PREV_INSURED_SHARE,PREV_LAST_APPL_IN_DAY_SHARE,PREV_CREDIT_TO_APPLICATION_MEDIAN,PREV_CREDIT_TO_APPLICATION_MAX,PREV_DOWN_PAYMENT_SHARE_MEDIAN,PREV_TOTAL_PAYMENT_TO_CREDIT_MEDIAN
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,
100001,3951.000,3951.000,24835.50,24835.5,23787.00,23787.0,2520.0,2520.0,24835.5,24835.5,...,-1740,-1740,8.0,8.0,0.000000,1.0,0.957782,0.957782,0.101468,1.328793
100002,9251.775,9251.775,179055.00,179055.0,179055.00,179055.0,0.0,0.0,179055.0,179055.0,...,-606,-606,24.0,24.0,0.000000,1.0,1.000000,1.000000,0.000000,1.240080
100003,64567.665,98356.995,337500.00,900000.0,348637.50,1035882.0,3442.5,6885.0,337500.0,900000.0,...,-746,-2341,12.0,12.0,0.666667,1.0,1.033000,1.150980,0.050029,1.139400
100004,5357.250,5357.250,24282.00,24282.0,20106.00,20106.0,4860.0,4860.0,24282.0,24282.0,...,-815,-815,4.0,4.0,0.000000,1.0,0.828021,0.828021,0.200148,1.065801
100005,4813.200,4813.200,22308.75,44617.5,20076.75,40153.5,4464.0,4464.0,44617.5,44617.5,...,-315,-757,12.0,12.0,0.000000,1.0,0.899950,0.899950,0.100050,1.438440


#### POS_CASH_balance_df

In [87]:
POS_CASH_balance_df = pd.read_csv(path_to_data+"POS_CASH_balance.csv")

In [88]:
POS_CASH_balance_df.info(show_counts=True)
POS_CASH_balance_df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10001358 entries, 0 to 10001357
Data columns (total 8 columns):
 #   Column                 Non-Null Count     Dtype  
---  ------                 --------------     -----  
 0   SK_ID_PREV             10001358 non-null  int64  
 1   SK_ID_CURR             10001358 non-null  int64  
 2   MONTHS_BALANCE         10001358 non-null  int64  
 3   CNT_INSTALMENT         9975287 non-null   float64
 4   CNT_INSTALMENT_FUTURE  9975271 non-null   float64
 5   NAME_CONTRACT_STATUS   10001358 non-null  object 
 6   SK_DPD                 10001358 non-null  int64  
 7   SK_DPD_DEF             10001358 non-null  int64  
dtypes: float64(2), int64(5), object(1)
memory usage: 610.4+ MB


(10001358, 8)

In [89]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
POS_CASH_balance_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("POS_CASH_balance.csv", case = False, na=False)]
display(POS_CASH_balance_description[["Row", "Description"]])


,Row,Description
142,SK_ID_PREV,ID of previous credit in Home Credit related t...
143,SK_ID_CURR,ID of loan in our sample
144,MONTHS_BALANCE,Month of balance relative to application date ...
145,CNT_INSTALMENT,Term of previous credit (can change over time)
146,CNT_INSTALMENT_FUTURE,Installments left to pay on the previous credit
147,NAME_CONTRACT_STATUS,Contract status during the month
148,SK_DPD,DPD (days past due) during the month of previo...
149,SK_DPD_DEF,DPD during the month with tolerance (debts wit...


In [90]:
categorical_cols_POS_CASH_bal_df = ["NAME_CONTRACT_STATUS"]
numerical_cols_POS_CASH_bal_df = ["CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE"]
special_cols_POS_CASH_bal_df = ["SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF"]

In [91]:
print("SPECIAL:", special_cols_POS_CASH_bal_df)
print("\nCATEGORICAL:", categorical_cols_POS_CASH_bal_df)
print("\nNUMERICAL:", numerical_cols_POS_CASH_bal_df)

SPECIAL: ['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE', 'SK_DPD', 'SK_DPD_DEF']

CATEGORICAL: ['NAME_CONTRACT_STATUS']

NUMERICAL: ['CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE']


In [93]:
POS_CASH_bal_work = POS_CASH_balance_df.copy()

In [97]:
POS_CASH_bal_work["POS_IS_OVERDUE"] = (POS_CASH_bal_work["SK_DPD"] > 0).astype(int)
POS_CASH_bal_work["POS_IS_OVERDUE_DEF"] = (POS_CASH_bal_work["SK_DPD_DEF"] > 0).astype(int)

In [100]:
POS_CASH_bal_work["POS_COMPLETION_RATIO"] = ((POS_CASH_bal_work["CNT_INSTALMENT"] - POS_CASH_bal_work["CNT_INSTALMENT_FUTURE"]) / POS_CASH_bal_work["CNT_INSTALMENT"].replace(0, np.nan))

In [101]:
pos_num_prev = agg_numerical_features(POS_CASH_bal_work, "SK_ID_PREV", numerical_cols_POS_CASH_bal_df, "POS")
pos_num_prev.head()

,POS_CNT_INSTALMENT_median,POS_CNT_INSTALMENT_max,POS_CNT_INSTALMENT_FUTURE_median,POS_CNT_INSTALMENT_FUTURE_max
SK_ID_PREV,,,,
1000001,12.0,12.0,11.0,12.0
1000002,6.0,6.0,2.0,4.0
1000003,12.0,12.0,10.5,12.0
1000004,10.0,10.0,6.5,10.0
1000005,10.0,10.0,5.0,10.0


In [102]:
pos_cat_prev = agg_categorical_features(POS_CASH_bal_work, "SK_ID_PREV", categorical_cols_POS_CASH_bal_df, "POS")
pos_cat_prev.head()

,POS_NAME_CONTRACT_STATUS_Active_COUNT,POS_NAME_CONTRACT_STATUS_Amortized debt_COUNT,POS_NAME_CONTRACT_STATUS_Approved_COUNT,POS_NAME_CONTRACT_STATUS_Canceled_COUNT,POS_NAME_CONTRACT_STATUS_Completed_COUNT,POS_NAME_CONTRACT_STATUS_Demand_COUNT,POS_NAME_CONTRACT_STATUS_Returned to the store_COUNT,POS_NAME_CONTRACT_STATUS_Signed_COUNT,POS_NAME_CONTRACT_STATUS_XNA_COUNT,POS_NAME_CONTRACT_STATUS_Active_SHARE,POS_NAME_CONTRACT_STATUS_Amortized debt_SHARE,POS_NAME_CONTRACT_STATUS_Approved_SHARE,POS_NAME_CONTRACT_STATUS_Canceled_SHARE,POS_NAME_CONTRACT_STATUS_Completed_SHARE,POS_NAME_CONTRACT_STATUS_Demand_SHARE,POS_NAME_CONTRACT_STATUS_Returned to the store_SHARE,POS_NAME_CONTRACT_STATUS_Signed_SHARE,POS_NAME_CONTRACT_STATUS_XNA_SHARE
SK_ID_PREV,,,,,,,,,,,,,,,,,,
1000001,2,0,0,0,1,0,0,0,0,0.666667,0.0,0.0,0.0,0.333333,0.0,0.0,0.0,0.0
1000002,4,0,0,0,1,0,0,0,0,0.800000,0.0,0.0,0.0,0.200000,0.0,0.0,0.0,0.0
1000003,4,0,0,0,0,0,0,0,0,1.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1000004,7,0,0,0,1,0,0,0,0,0.875000,0.0,0.0,0.0,0.125000,0.0,0.0,0.0,0.0
1000005,10,0,0,0,1,0,0,0,0,0.909091,0.0,0.0,0.0,0.090909,0.0,0.0,0.0,0.0


In [103]:
pos_special_prev = (
    POS_CASH_bal_work
    .groupby("SK_ID_PREV")
    .agg(
        POS_MONTH_COUNT=(
            "MONTHS_BALANCE",
            "count"
        ),

        POS_OLDEST_MONTH=(
            "MONTHS_BALANCE",
            "min"
        ),

        POS_MOST_RECENT_MONTH=(
            "MONTHS_BALANCE",
            "max"
        ),

        POS_MAX_DPD=(
            "SK_DPD",
            "max"
        ),

        POS_MAX_DPD_DEF=(
            "SK_DPD_DEF",
            "max"
        ),

        POS_OVERDUE_MONTH_COUNT=(
            "POS_IS_OVERDUE",
            "sum"
        ),

        POS_OVERDUE_DEF_MONTH_COUNT=(
            "POS_IS_OVERDUE_DEF",
            "sum"
        ),

        POS_COMPLETION_RATIO_MEDIAN=(
            "POS_COMPLETION_RATIO",
            "median"
        ),

        POS_COMPLETION_RATIO_MAX=(
            "POS_COMPLETION_RATIO",
            "max"
        )
    )
)

pos_special_prev.head()

,POS_MONTH_COUNT,POS_OLDEST_MONTH,POS_MOST_RECENT_MONTH,POS_MAX_DPD,POS_MAX_DPD_DEF,POS_OVERDUE_MONTH_COUNT,POS_OVERDUE_DEF_MONTH_COUNT,POS_COMPLETION_RATIO_MEDIAN,POS_COMPLETION_RATIO_MAX
SK_ID_PREV,,,,,,,,,
1000001,3,-10,-8,0,0,0,0,0.083333,1.00
1000002,5,-54,-50,0,0,0,0,0.666667,1.00
1000003,4,-4,-1,0,0,0,0,0.125000,0.25
1000004,8,-29,-22,0,0,0,0,0.350000,1.00
1000005,11,-56,-46,0,0,0,0,0.500000,1.00


In [104]:
pos_special_prev["POS_HISTORY_LENGTH"] = (
    pos_special_prev["POS_MOST_RECENT_MONTH"]
    -
    pos_special_prev["POS_OLDEST_MONTH"]
)

In [105]:
pos_features_prev = pd.concat(
    [
        pos_num_prev,
        pos_cat_prev,
        pos_special_prev
    ],
    axis=1
)

pos_features_prev.head()

,POS_CNT_INSTALMENT_median,POS_CNT_INSTALMENT_max,POS_CNT_INSTALMENT_FUTURE_median,POS_CNT_INSTALMENT_FUTURE_max,POS_NAME_CONTRACT_STATUS_Active_COUNT,POS_NAME_CONTRACT_STATUS_Amortized debt_COUNT,POS_NAME_CONTRACT_STATUS_Approved_COUNT,POS_NAME_CONTRACT_STATUS_Canceled_COUNT,POS_NAME_CONTRACT_STATUS_Completed_COUNT,POS_NAME_CONTRACT_STATUS_Demand_COUNT,...,POS_MONTH_COUNT,POS_OLDEST_MONTH,POS_MOST_RECENT_MONTH,POS_MAX_DPD,POS_MAX_DPD_DEF,POS_OVERDUE_MONTH_COUNT,POS_OVERDUE_DEF_MONTH_COUNT,POS_COMPLETION_RATIO_MEDIAN,POS_COMPLETION_RATIO_MAX,POS_HISTORY_LENGTH
SK_ID_PREV,,,,,,,,,,,,,,,,,,,,,
1000001,12.0,12.0,11.0,12.0,2,0,0,0,1,0,...,3,-10,-8,0,0,0,0,0.083333,1.00,2
1000002,6.0,6.0,2.0,4.0,4,0,0,0,1,0,...,5,-54,-50,0,0,0,0,0.666667,1.00,4
1000003,12.0,12.0,10.5,12.0,4,0,0,0,0,0,...,4,-4,-1,0,0,0,0,0.125000,0.25,3
1000004,10.0,10.0,6.5,10.0,7,0,0,0,1,0,...,8,-29,-22,0,0,0,0,0.350000,1.00,7
1000005,10.0,10.0,5.0,10.0,10,0,0,0,1,0,...,11,-56,-46,0,0,0,0,0.500000,1.00,10


In [106]:
print(
    "Размер:",
    pos_features_prev.shape
)

print(
    "SK_ID_PREV уникален:",
    pos_features_prev.index.is_unique
)

Размер: (936325, 32)
SK_ID_PREV уникален: True


In [107]:
prev_id_map = (
    previous_application_df[
        [
            "SK_ID_PREV",
            "SK_ID_CURR"
        ]
    ]
    .copy()
)

In [108]:
pos_with_client = (
    pos_features_prev
    .reset_index()
    .merge(
        prev_id_map,
        on="SK_ID_PREV",
        how="left",
        validate="1:1"
    )
)

pos_with_client.head()

,SK_ID_PREV,POS_CNT_INSTALMENT_median,POS_CNT_INSTALMENT_max,POS_CNT_INSTALMENT_FUTURE_median,POS_CNT_INSTALMENT_FUTURE_max,POS_NAME_CONTRACT_STATUS_Active_COUNT,POS_NAME_CONTRACT_STATUS_Amortized debt_COUNT,POS_NAME_CONTRACT_STATUS_Approved_COUNT,POS_NAME_CONTRACT_STATUS_Canceled_COUNT,POS_NAME_CONTRACT_STATUS_Completed_COUNT,...,POS_OLDEST_MONTH,POS_MOST_RECENT_MONTH,POS_MAX_DPD,POS_MAX_DPD_DEF,POS_OVERDUE_MONTH_COUNT,POS_OVERDUE_DEF_MONTH_COUNT,POS_COMPLETION_RATIO_MEDIAN,POS_COMPLETION_RATIO_MAX,POS_HISTORY_LENGTH,SK_ID_CURR
0,1000001,12.0,12.0,11.0,12.0,2,0,0,0,1,...,-10,-8,0,0,0,0,0.083333,1.00,2,158271.0
1,1000002,6.0,6.0,2.0,4.0,4,0,0,0,1,...,-54,-50,0,0,0,0,0.666667,1.00,4,101962.0
2,1000003,12.0,12.0,10.5,12.0,4,0,0,0,0,...,-4,-1,0,0,0,0,0.125000,0.25,3,252457.0
3,1000004,10.0,10.0,6.5,10.0,7,0,0,0,1,...,-29,-22,0,0,0,0,0.350000,1.00,7,260094.0
4,1000005,10.0,10.0,5.0,10.0,10,0,0,0,1,...,-56,-46,0,0,0,0,0.500000,1.00,10,176456.0


In [109]:
print(
    "POS договоров без SK_ID_CURR:",
    pos_with_client["SK_ID_CURR"].isna().sum()
)

POS договоров без SK_ID_CURR: 37422


In [110]:
pos_mapped = (
    pos_with_client[
        pos_with_client["SK_ID_CURR"].notna()
    ]
    .copy()
)

pos_unmapped = (
    pos_with_client[
        pos_with_client["SK_ID_CURR"].isna()
    ]
    .copy()
)

In [111]:
assert (
    len(pos_mapped)
    +
    len(pos_unmapped)
    ==
    len(pos_with_client)
)

In [112]:
pos_count_cols = [
    col
    for col in pos_mapped.columns
    if col.endswith("_COUNT")
]

In [113]:
pos_client_counts = (
    pos_mapped
    .groupby("SK_ID_CURR")[
        pos_count_cols
    ]
    .sum()
)

pos_client_counts.head()

,POS_NAME_CONTRACT_STATUS_Active_COUNT,POS_NAME_CONTRACT_STATUS_Amortized debt_COUNT,POS_NAME_CONTRACT_STATUS_Approved_COUNT,POS_NAME_CONTRACT_STATUS_Canceled_COUNT,POS_NAME_CONTRACT_STATUS_Completed_COUNT,POS_NAME_CONTRACT_STATUS_Demand_COUNT,POS_NAME_CONTRACT_STATUS_Returned to the store_COUNT,POS_NAME_CONTRACT_STATUS_Signed_COUNT,POS_NAME_CONTRACT_STATUS_XNA_COUNT,POS_MONTH_COUNT,POS_OVERDUE_MONTH_COUNT,POS_OVERDUE_DEF_MONTH_COUNT
SK_ID_CURR,,,,,,,,,,,,
100001.0,4,0,0,0,1,0,0,0,0,5,0,0
100002.0,19,0,0,0,0,0,0,0,0,19,0,0
100003.0,26,0,0,0,2,0,0,0,0,28,0,0
100004.0,3,0,0,0,1,0,0,0,0,4,0,0
100005.0,9,0,0,0,1,0,0,1,0,11,0,0


In [115]:
pos_client_counts[
    "POS_OVERDUE_MONTH_SHARE"
] = (
    pos_client_counts[
        "POS_OVERDUE_MONTH_COUNT"
    ]
    /
    pos_client_counts[
        "POS_MONTH_COUNT"
    ].replace(0, np.nan)
)

pos_client_counts[
    "POS_OVERDUE_DEF_MONTH_SHARE"
] = (
    pos_client_counts[
        "POS_OVERDUE_DEF_MONTH_COUNT"
    ]
    /
    pos_client_counts[
        "POS_MONTH_COUNT"
    ].replace(0, np.nan)
)

In [116]:
pos_status_count_cols = [
    col
    for col in pos_client_counts.columns
    if col.startswith(
        "POS_NAME_CONTRACT_STATUS_"
    )
    and col.endswith("_COUNT")
]

In [117]:
for col in pos_status_count_cols:

    share_col = col.replace(
        "_COUNT",
        "_SHARE"
    )

    pos_client_counts[
        share_col
    ] = (
        pos_client_counts[col]
        /
        pos_client_counts[
            "POS_MONTH_COUNT"
        ].replace(0, np.nan)
    )

In [118]:
pos_client_special = (
    pos_mapped
    .groupby("SK_ID_CURR")
    .agg(
        POS_CONTRACT_WITH_HISTORY_COUNT=(
            "SK_ID_PREV",
            "nunique"
        ),

        POS_HISTORY_LENGTH_MAX=(
            "POS_HISTORY_LENGTH",
            "max"
        ),

        POS_HISTORY_LENGTH_MEDIAN=(
            "POS_HISTORY_LENGTH",
            "median"
        ),

        POS_OLDEST_MONTH_CLIENT=(
            "POS_OLDEST_MONTH",
            "min"
        ),

        POS_MOST_RECENT_MONTH_CLIENT=(
            "POS_MOST_RECENT_MONTH",
            "max"
        ),

        POS_MAX_DPD_CLIENT=(
            "POS_MAX_DPD",
            "max"
        ),

        POS_MAX_DPD_DEF_CLIENT=(
            "POS_MAX_DPD_DEF",
            "max"
        ),

        POS_COMPLETION_RATIO_MEDIAN_CLIENT=(
            "POS_COMPLETION_RATIO_MEDIAN",
            "median"
        ),

        POS_COMPLETION_RATIO_MAX_CLIENT=(
            "POS_COMPLETION_RATIO_MAX",
            "max"
        )
    )
)

In [119]:
pos_numeric_prev_cols = list(
    pos_num_prev.columns
)

In [120]:
pos_client_num = (
    pos_mapped
    .groupby("SK_ID_CURR")[
        pos_numeric_prev_cols
    ]
    .agg(["median", "max"])
)

In [121]:
pos_client_num.columns = [
    col + "_" + func
    for col, func
    in pos_client_num.columns
]

In [122]:
pos_features_client = pd.concat(
    [
        pos_client_counts,
        pos_client_special,
        pos_client_num
    ],
    axis=1
)

pos_features_client.head()

,POS_NAME_CONTRACT_STATUS_Active_COUNT,POS_NAME_CONTRACT_STATUS_Amortized debt_COUNT,POS_NAME_CONTRACT_STATUS_Approved_COUNT,POS_NAME_CONTRACT_STATUS_Canceled_COUNT,POS_NAME_CONTRACT_STATUS_Completed_COUNT,POS_NAME_CONTRACT_STATUS_Demand_COUNT,POS_NAME_CONTRACT_STATUS_Returned to the store_COUNT,POS_NAME_CONTRACT_STATUS_Signed_COUNT,POS_NAME_CONTRACT_STATUS_XNA_COUNT,POS_MONTH_COUNT,...,POS_COMPLETION_RATIO_MEDIAN_CLIENT,POS_COMPLETION_RATIO_MAX_CLIENT,POS_CNT_INSTALMENT_median_median,POS_CNT_INSTALMENT_median_max,POS_CNT_INSTALMENT_max_median,POS_CNT_INSTALMENT_max_max,POS_CNT_INSTALMENT_FUTURE_median_median,POS_CNT_INSTALMENT_FUTURE_median_max,POS_CNT_INSTALMENT_FUTURE_max_median,POS_CNT_INSTALMENT_FUTURE_max_max
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,
100001.0,4,0,0,0,1,0,0,0,0,5,...,0.500000,1.00,4.0,4.0,4.0,4.0,2.0,2.0,4.0,4.0
100002.0,19,0,0,0,0,0,0,0,0,19,...,0.375000,0.75,24.0,24.0,24.0,24.0,15.0,15.0,24.0,24.0
100003.0,26,0,0,0,2,0,0,0,0,28,...,0.458333,1.00,12.0,12.0,12.0,12.0,6.5,8.5,12.0,12.0
100004.0,3,0,0,0,1,0,0,0,0,4,...,0.375000,1.00,4.0,4.0,4.0,4.0,2.5,2.5,4.0,4.0
100005.0,9,0,0,0,1,0,0,1,0,11,...,0.375000,1.00,12.0,12.0,12.0,12.0,7.5,7.5,12.0,12.0


In [123]:
print(
    "Размер:",
    pos_features_client.shape
)

print(
    "SK_ID_CURR уникален:",
    pos_features_client.index.is_unique
)

print(
    "Пропусков в ID:",
    pos_features_client.index.isna().sum()
)

Размер: (334359, 40)
SK_ID_CURR уникален: True
Пропусков в ID: 0


#### installments_payments_df

In [ ]:
installments_payments_df = pd.read_csv(path_to_data+"installments_payments.csv")

In [ ]:
installments_payments_df.info(show_counts=True)
installments_payments_df.shape

In [ ]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
inst_payment_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("installments_payments.csv", case = False, na=False)]
display(inst_payment_description[["Row", "Description"]])

In [ ]:
categorical_cols_inst_payment_df = []
numerical_cols_inst_payment_df = ["AMT_INSTALMENT", "AMT_PAYMENT"]
special_cols_inst_payment_df = ["SK_ID_PREV", "SK_ID_CURR", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT"]

In [ ]:
print("SPECIAL:", special_cols_inst_payment_df)
print("\nCATEGORICAL:", categorical_cols_inst_payment_df)
print("\nNUMERICAL:", numerical_cols_inst_payment_df)

#### credit_card_balance_df

In [ ]:
credit_card_balance_df = pd.read_csv(path_to_data+"credit_card_balance.csv")

In [ ]:
credit_card_balance_df.info(show_counts=True)
credit_card_balance_df.shape

In [ ]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
credit_card_balance_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("credit_card_balance.csv", case = False, na=False)]
display(credit_card_balance_description[["Row", "Description"]])

In [ ]:
categorical_cols_credit_card_balance_df = ["NAME_CONTRACT_STATUS"]
numerical_cols_credit_card_balance_df = ["AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_ATM_CURRENT", "AMT_DRAWINGS_CURRENT", "AMT_DRAWINGS_OTHER_CURRENT", "AMT_DRAWINGS_POS_CURRENT", "AMT_INST_MIN_REGULARITY", "AMT_PAYMENT_CURRENT", "AMT_PAYMENT_TOTAL_CURRENT", "AMT_RECEIVABLE_PRINCIPAL", "AMT_RECIVABLE", "AMT_TOTAL_RECEIVABLE", "CNT_DRAWINGS_ATM_CURRENT", "CNT_DRAWINGS_CURRENT", "CNT_DRAWINGS_OTHER_CURRENT", "CNT_DRAWINGS_POS_CURRENT", "CNT_INSTALMENT_MATURE_CUM"]
special_cols_credit_card_balance_df = ["SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF"]

In [ ]:
print("SPECIAL:", special_cols_credit_card_balance_df)
print("\nCATEGORICAL:", categorical_cols_credit_card_balance_df)
print("\nNUMERICAL:", numerical_cols_credit_card_balance_df)

### Save final tables

In [69]:
save_path = "/home/usl/PycharmProjects/home-credit-default-risk/data/processed/"

In [70]:
application_train_df.to_csv(save_path+"application_train.csv", index=False)
application_train_df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,0,0,0,0,1.0,0.0,0.0,1.0,0.0,1.0
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [71]:
application_test_df.to_csv(save_path+"application_test.csv", index=False)
application_test_df

,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
2,100013,Cash loans,M,Y,Y,0,202500.0,663264.0,69777.0,630000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0
3,100028,Cash loans,F,N,Y,2,315000.0,1575000.0,49018.5,1575000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
4,100038,Cash loans,M,Y,N,1,180000.0,625500.0,32067.0,625500.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48739,456221,Cash loans,F,N,Y,0,121500.0,412560.0,17473.5,270000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
48740,456222,Cash loans,F,N,N,2,157500.0,622413.0,31909.5,495000.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
48741,456223,Cash loans,F,Y,Y,1,202500.0,315000.0,33205.5,315000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,3.0,1.0
48742,456224,Cash loans,M,N,N,0,225000.0,450000.0,25128.0,450000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,2.0


Split into train and val

In [ ]:
train_users = application_train_df["SK_ID_CURR"].unique()
train_ids, val_ids = train_test_split(train_users, test_size=0.2, random_state=42)
train_ids

In [ ]:
train_df = application_train_df[application_train_df.SK_ID_CURR.isin(train_ids)].copy()
val_df = application_train_df[application_train_df.SK_ID_CURR.isin(val_ids)].copy()

In [ ]:
train_df.shape, val_df.shape

In [ ]:
application_train_df["TARGET"].mean(), train_df["TARGET"].mean(), val_df["TARGET"].mean()

In [ ]:
drop_feat = ["SK_ID_CURR"]
train_df.drop(drop_feat, axis=1, inplace=True)
val_df.drop(drop_feat, axis=1, inplace=True)
application_test_df.drop(drop_feat, axis=1, inplace=True)

In [ ]:
X_train = train_df.drop("TARGET", axis=1)
y_train = train_df["TARGET"]
X_val = val_df.drop("TARGET", axis=1)
y_val = val_df["TARGET"]

In [ ]:
cat_cols = X_train.select_dtypes(include = 'object').columns.tolist()
len(cat_cols)

In [ ]:
for col in cat_cols:
    categories = X_train[col].dropna().unique()
    X_train[col] = pd.Categorical(X_train[col], categories = categories)
    X_val[col] = pd.Categorical(X_val[col], categories = categories)


In [ ]:
X_train.select_dtypes(include='object').columns

### Save final tables

### LightGBM

In [ ]:
lgb_clf = lgb.LGBMClassifier(n_estimators=1000, max_depth=4, learning_rate=0.1, random_state = 42, n_jobs=-1)
lgb_clf.fit(X_train, y_train, eval_set = [(X_val, y_val)], eval_metric='auc', categorical_feature=cat_cols, callbacks = [early_stopping(stopping_rounds=100), log_evaluation(period = 100)])

threshold = 0.5
val_pred_proba = lgb_clf.predict_proba(X_val)[:, 1]
val_pred = (val_pred_proba >= threshold).astype(int)


roc_auc_lgb = roc_auc_score(y_val, val_pred_proba)
prec_lgb = precision_score(y_val, val_pred, zero_division=0)
rec_lgb = recall_score(y_val, val_pred, zero_division=0)

print("Best_iteration:", lgb_clf.best_iteration_)
print("ROC-AUC_lgb:", roc_auc_lgb)
print("Precision_lgb:", prec_lgb)
print("Recall_lgb:", rec_lgb)

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_val, val_pred)
plt.title("Confusion Matrix LightGBM")

And now we will calculate the roc_curve and plot a graph:

In [ ]:
fpr, tpr, thresholds = roc_curve(y_val, val_pred_proba)
plt.figure(figsize=(7, 6))

plt.plot(fpr, tpr, label = f"LightGBM ROC-AUC={roc_auc_lgb:.4f}")
plt.plot([0, 1], [0, 1], 'k--', label = "Random classifier ROC-AUC=0.5")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve LightGBM")
plt.legend(loc="lower right")
plt.grid()

We will prepare the test part for training:

In [ ]:
application_test_df.shape

In [ ]:
X_test = application_test_df

In [ ]:
for col in cat_cols:
    X_test[col] = pd.Categorical(X_test[col], categories = X_train[col].cat.categories)
X_test.shape, X_train.shape

In [ ]:
list(X_train.columns) == list(X_test.columns)

In [ ]:
test_pred_proba = lgb_clf.predict_proba(X_test)[:, 1]

In [ ]:
submission = sample_submission_df.copy()
submission["TARGET"] = test_pred_proba
submission.head()

In [ ]:
submission.shape, application_test_df.shape

In [ ]:
submission.to_csv("submission_lgb_baseline.csv", index=False)